
# 02 — Paired context-robustness controls

> **Provenance note.** This is a cleaned **reference/reproducibility implementation reconstructed from the protocol documented in the paper**. It is not claimed to be the exact historical execution notebook. Replace it with the original executed robustness notebook when that file is available.

This notebook implements the SylFishBD paired interventions used with frozen BioCLIP2 and the four-template scientific-name ensemble: weak/medium/strong background blur, gray/mean-color/white replacement, tight crop, background-only inpainting, and cross-species background swap. The statistical helpers use paired inference with seed 42.


In [ ]:

from pathlib import Path
import random, math
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import cv2
import torch
import open_clip
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from scipy.stats import binomtest, chi2

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


## Configuration
Set image and binary-mask roots. `build_pairs` matches masks by filename stem and infers the class from the image path.

In [ ]:

IMAGE_ROOT=Path('/path/to/SylFishBD/images')
MASK_ROOT=Path('/path/to/SylFishBD/masks')
OUT_DIR=Path('../results/generated_context')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES=['Rui','Katla','Mrigal','Tilapia','Pabda','Ilish','Koi']
EXPECTED={'Rui':1670,'Katla':1133,'Mrigal':1293,'Tilapia':1326,'Pabda':862,'Ilish':789,'Koi':592}
SCI={'Rui':'Labeo rohita','Katla':'Catla catla','Mrigal':'Cirrhinus cirrhosus','Tilapia':'Oreochromis niloticus','Pabda':'Ompok pabda','Ilish':'Tenualosa ilisha','Koi':'Anabas testudineus'}
TEMPLATES=['a photo of {name}, a fish species','an image of {name}, a fish species',
           'a photograph of {name}, a fish species','a specimen of {name}, a fish species']
EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}

def infer_class(p):
    low=[x.lower() for x in p.parts]
    for c in CLASSES:
        if c.lower() in low: return c
    for c in CLASSES:
        if any(c.lower() in x for x in low): return c
    return None

def build_pairs():
    mask_index={p.stem:p for p in MASK_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in EXTS}
    rows=[]
    for p in sorted(IMAGE_ROOT.rglob('*')):
        if not (p.is_file() and p.suffix.lower() in EXTS): continue
        cls=infer_class(p.relative_to(IMAGE_ROOT))
        m=mask_index.get(p.stem)
        if cls and m: rows.append((cls,str(p),str(m)))
    df=pd.DataFrame(rows,columns=['label','image_path','mask_path'])
    print('pairs:',len(df)); print(df.label.value_counts().reindex(CLASSES))
    return df

pairs=build_pairs()
assert pairs.label.value_counts().to_dict()==EXPECTED, 'Pair counts do not match paper subset.'


## Intervention functions

In [ ]:

def read_rgb(path):
    return cv2.cvtColor(cv2.imread(str(path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

def read_mask(path, shape):
    m=cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if m is None: raise FileNotFoundError(path)
    if m.shape[:2] != shape[:2]: m=cv2.resize(m,(shape[1],shape[0]),interpolation=cv2.INTER_NEAREST)
    return m>127

def composite(fg, bg, mask):
    out=bg.copy(); out[mask]=fg[mask]; return out

def blur_background(img, mask, sigma_ratio):
    sigma=max(0.1, sigma_ratio*min(img.shape[:2]))
    k=max(3, int(round(sigma*6))|1)
    bg=cv2.GaussianBlur(img,(k,k),sigmaX=sigma,sigmaY=sigma)
    return composite(img,bg,mask)

def constant_background(img, mask, mode):
    if mode=='white': val=np.array([255,255,255],dtype=np.uint8)
    elif mode=='gray': val=np.array([128,128,128],dtype=np.uint8)
    elif mode=='mean':
        bgpix=img[~mask]
        val=np.rint(bgpix.mean(axis=0) if len(bgpix) else img.reshape(-1,3).mean(axis=0)).astype(np.uint8)
    else: raise ValueError(mode)
    bg=np.empty_like(img); bg[:]=val
    return composite(img,bg,mask)

def tight_crop(img, mask, expand=0.10):
    ys,xs=np.where(mask)
    if len(xs)==0: return img
    x0,x1=xs.min(),xs.max(); y0,y1=ys.min(),ys.max()
    pad_x=int(round((x1-x0+1)*expand)); pad_y=int(round((y1-y0+1)*expand))
    x0=max(0,x0-pad_x); x1=min(img.shape[1]-1,x1+pad_x)
    y0=max(0,y0-pad_y); y1=min(img.shape[0]-1,y1+pad_y)
    return img[y0:y1+1,x0:x1+1]

def inpaint_foreground(img, mask):
    # Slight dilation before Telea inpainting, as a background-only negative control.
    ker=np.ones((5,5),np.uint8)
    dil=cv2.dilate(mask.astype(np.uint8)*255,ker,iterations=2)
    bgr=cv2.cvtColor(img,cv2.COLOR_RGB2BGR)
    out=cv2.inpaint(bgr,dil,3,cv2.INPAINT_TELEA)
    return cv2.cvtColor(out,cv2.COLOR_BGR2RGB)


## BioCLIP2 classifier

In [ ]:

model_name='hf-hub:imageomics/bioclip-2'
model, preprocess = open_clip.create_model_from_pretrained(model_name, device=DEVICE)
tokenizer=open_clip.get_tokenizer(model_name)
model.eval()

@torch.inference_mode()
def text_prototypes():
    protos=[]
    for c in CLASSES:
        prompts=[t.format(name=SCI[c]) for t in TEMPLATES]
        z=model.encode_text(tokenizer(prompts).to(DEVICE)); z=z/z.norm(dim=-1,keepdim=True)
        z=z.mean(0); z=z/z.norm(); protos.append(z.cpu())
    return torch.stack(protos).numpy()
PROTO=text_prototypes()

@torch.inference_mode()
def encode_pil_batch(pils, batch_size=64):
    zs=[]
    for s in range(0,len(pils),batch_size):
        x=torch.stack([preprocess(im) for im in pils[s:s+batch_size]]).to(DEVICE)
        z=model.encode_image(x); z=z/z.norm(dim=-1,keepdim=True); zs.append(z.cpu())
    return torch.cat(zs).numpy()

def pred_from_pils(pils):
    emb=encode_pil_batch(pils)
    return np.asarray(CLASSES)[np.argmax(emb@PROTO.T,axis=1)]


## Generate paired conditions
For memory safety, a production run should stream/batch images and optionally cache transformed images/embeddings. The compact implementation below is easy to audit.

In [ ]:

CONDITIONS=['raw','weak_blur','medium_blur','strong_blur','gray_mask','mean_color_mask','white_mask','tight_crop','background_only']

def make_condition(img,mask,name):
    if name=='raw': return img
    if name=='weak_blur': return blur_background(img,mask,0.01)
    if name=='medium_blur': return blur_background(img,mask,0.03)
    if name=='strong_blur': return blur_background(img,mask,0.06)
    if name=='gray_mask': return constant_background(img,mask,'gray')
    if name=='mean_color_mask': return constant_background(img,mask,'mean')
    if name=='white_mask': return constant_background(img,mask,'white')
    if name=='tight_crop': return tight_crop(img,mask,0.10)
    if name=='background_only': return inpaint_foreground(img,mask)
    raise ValueError(name)

def evaluate_condition(name, batch_size=64):
    preds=[]
    for s in tqdm(range(0,len(pairs),batch_size),desc=name):
        batch=pairs.iloc[s:s+batch_size]
        pils=[]
        for r in batch.itertuples():
            img=read_rgb(r.image_path); mask=read_mask(r.mask_path,img.shape)
            arr=make_condition(img,mask,name)
            pils.append(Image.fromarray(arr))
        preds.extend(pred_from_pils(pils))
    return np.asarray(preds)

preds={name:evaluate_condition(name) for name in CONDITIONS}
y=pairs.label.to_numpy()


## Paired statistical helpers

In [ ]:

def basic_metrics(pred):
    return dict(accuracy_pct=100*accuracy_score(y,pred),
                balanced_accuracy_pct=100*balanced_accuracy_score(y,pred),
                macro_f1_pct=100*f1_score(y,pred,average='macro'))

def paired_bootstrap_delta(pred_a,pred_b,n_boot=2000,seed=42):
    rng=np.random.default_rng(seed); deltas=[]
    labels=np.asarray(y); classes=np.unique(labels)
    idx_by={c:np.flatnonzero(labels==c) for c in classes}
    ca=(pred_a==labels); cb=(pred_b==labels)
    for _ in range(n_boot):
        idx=np.concatenate([rng.choice(v,size=len(v),replace=True) for v in idx_by.values()])
        deltas.append(100*(cb[idx].mean()-ca[idx].mean()))
    return np.percentile(deltas,[2.5,97.5])

def mcnemar_exact(pred_a,pred_b):
    ca=pred_a==y; cb=pred_b==y
    n01=int(np.sum(~ca & cb)); n10=int(np.sum(ca & ~cb))
    p=binomtest(min(n01,n10),n=n01+n10,p=0.5,alternative='two-sided').pvalue if n01+n10 else 1.0
    return n01,n10,p

def bh_fdr(pvals):
    p=np.asarray(pvals,float); n=len(p); order=np.argsort(p); q=np.empty(n); prev=1.0
    for rank_i in range(n-1,-1,-1):
        i=order[rank_i]; rank=rank_i+1; prev=min(prev,p[i]*n/rank); q[i]=prev
    return np.minimum(q,1.0)

def cochran_q(correct_matrix):
    # matrix: n samples x k paired binary outcomes
    X=np.asarray(correct_matrix,dtype=float); n,k=X.shape
    col=X.sum(0); row=X.sum(1); total=col.sum()
    denom=k*total-np.sum(row**2)
    Q=(k-1)*(k*np.sum(col**2)-total**2)/denom if denom>0 else 0.0
    return Q, 1-chi2.cdf(Q,k-1)


## Aggregate context ladder

In [ ]:

raw=preds['raw']; rows=[]; pvals=[]
for name in CONDITIONS:
    m=basic_metrics(preds[name])
    if name=='raw':
        rows.append({'condition':name,**m,'delta_pp':0.0,'ci_low_pp':np.nan,'ci_high_pp':np.nan,'mcnemar_p':np.nan})
    else:
        delta=100*((preds[name]==y).mean()-(raw==y).mean())
        lo,hi=paired_bootstrap_delta(raw,preds[name])
        _,_,p=mcnemar_exact(raw,preds[name]); pvals.append(p)
        rows.append({'condition':name,**m,'delta_pp':delta,'ci_low_pp':lo,'ci_high_pp':hi,'mcnemar_p':p})
ladder=pd.DataFrame(rows)
q=bh_fdr(pvals); ladder.loc[ladder.condition!='raw','fdr_q']=q
ladder.to_csv(OUT_DIR/'context_ladder_generated.csv',index=False)
display(ladder)

correct=np.column_stack([preds[c]==y for c in CONDITIONS[:7]])
print('Cochran Q (raw through white mask):', cochran_q(correct))


## Cross-species background swap
A deterministic donor is selected from another class. The recipient foreground is composited onto an inpainted donor background; donor-follow can then be compared with a label-permutation null.

In [ ]:

rng=np.random.default_rng(SEED)
indices_by={c:np.flatnonzero(y==c) for c in CLASSES}
donor_idx=[]
for i,c in enumerate(y):
    other=[x for x in CLASSES if x!=c]
    dc=other[i % len(other)]
    pool=indices_by[dc]
    donor_idx.append(pool[(i//len(other)) % len(pool)])
donor_idx=np.asarray(donor_idx)
donor_labels=y[donor_idx]

swap_preds=[]; B=32
for s in tqdm(range(0,len(pairs),B),desc='background_swap'):
    pils=[]
    for i in range(s,min(s+B,len(pairs))):
        r=pairs.iloc[i]; d=pairs.iloc[donor_idx[i]]
        img=read_rgb(r.image_path); mask=read_mask(r.mask_path,img.shape)
        dimg=read_rgb(d.image_path); dmask=read_mask(d.mask_path,dimg.shape)
        dbg=inpaint_foreground(dimg,dmask)
        dbg=cv2.resize(dbg,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_LINEAR)
        arr=composite(img,dbg,mask); pils.append(Image.fromarray(arr))
    swap_preds.extend(pred_from_pils(pils))
swap_preds=np.asarray(swap_preds)
print('swap accuracy %',100*accuracy_score(y,swap_preds))
print('donor-follow %',100*np.mean(swap_preds==donor_labels))

# Permutation null for donor-follow; the paper used 5,000 permutations.
rng=np.random.default_rng(SEED)
obs=np.mean(swap_preds==donor_labels)
null=np.array([np.mean(swap_preds==rng.permutation(donor_labels)) for _ in range(5000)])
p_one_sided=(np.sum(null>=obs)+1)/(len(null)+1)
print('null mean %',100*null.mean(),'one-sided p',p_one_sided)



## Paper-value cross-check

Compare the generated primary ladder with `../results/context_ladder.csv`. The paper reports the paired SylFishBD BioCLIP2 scientific-ensemble ladder. Any mismatch should be traced to mask pairing, preprocessing, model/checkpoint version, or transformation details rather than manually edited to match the paper.
